In [ ]:
# ==============================================================================
# BƯỚC 3 v12-HierOnly – Hierarchical Pooling (Ward's method)
# Method: gộp N token query thành K = max(1, round(N * ratio)) cụm
#         bằng Ward agglomerative clustering (cosine distance),
#         mean pool các vector trong mỗi cụm → K vectors đại diện,
#         MaxSim K vectors đó với doc → sum scores.
# Ablation: thử các topk_ratio khác nhau.
# Baseline: traditional MaxSim (sum toàn bộ token).
# ==============================================================================

print(">>> BƯỚC 3 v12-HierOnly: Hierarchical Pooling (Ward's method)")

import torch
import torch.nn.functional as F
import numpy as np
import json, os, pickle, gc
import pandas as pd
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
TOPK_RATIOS      = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50

# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# SHARED HELPERS
# ==============================================================================
def build_doc_matrix(docs_emb, device):
    n     = len(docs_emb)
    max_l = max(d.shape[0] for d in docs_emb)
    D     = docs_emb[0].shape[1]
    doc_matrix = torch.zeros(n, max_l, D, device=device)
    doc_mask   = torch.zeros(n, max_l,    device=device, dtype=torch.bool)
    for i, d in enumerate(docs_emb):
        L = d.shape[0]
        doc_matrix[i, :L] = F.normalize(d.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """q_norm: (Q, D) → returns (Q, n_docs)"""
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values


# ==============================================================================
# WARD AGGLOMERATIVE CLUSTERING (cosine distance)
# Input : vecs (N, D) đã L2-normalize
# Output: centroids (K, D) — mean pool mỗi cụm rồi L2-normalize
# ==============================================================================
def _ward_distance_matrix(cents, sizes):
    """Tính Ward distance giữa các cụm hiện tại."""
    device  = cents.device
    sim     = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist = 2.0 * (1.0 - sim)          # cosine → squared euclidean trên unit sphere
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    # Mask tam giác dưới + đường chéo (chỉ xét upper triangle)
    mask = torch.ones(cents.shape[0], cents.shape[0], dtype=torch.bool, device=device).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    """
    Ward agglomerative clustering, merge xuống còn n_clusters.
    Trả về centroids (K, D) đã L2-normalize (mean pool mỗi cụm).
    """
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()

    device = vecs.device
    sums   = vecs.clone().float()          # tổng vector trong mỗi cụm
    sizes  = torch.ones(N, device=device, dtype=torch.float32)
    cents  = F.normalize(sums, dim=-1)    # centroid = mean pool → normalize
    active = torch.ones(N, dtype=torch.bool, device=device)
    C      = N

    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act = cents[active_idx]
        s_act = sizes[active_idx]

        ward     = _ward_distance_matrix(c_act, s_act)
        flat_idx = ward.argmin().item()
        n_act    = active_idx.shape[0]
        ai       = flat_idx // n_act
        aj       = flat_idx %  n_act
        gi       = active_idx[ai].item()
        gj       = active_idx[aj].item()

        # Merge gj vào gi
        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1

    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]   # (K, D)


# ==============================================================================
# WARD POOL SCORES — tất cả ratio trong 1 lần (incremental merge)
# Sort ratio descending: ít merge trước → nhiều merge sau
# ==============================================================================
def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    """
    q_norm     : (N, D) đã L2-normalize — token hợp lệ của query
    topk_ratios: list[float]
    Returns    : dict {ratio: scores (n_docs,)}
    """
    N       = q_norm.shape[0]
    results = {}

    # Descending → ratio lớn (giữ nhiều token) merge ít → ratio nhỏ merge nhiều
    sorted_ratios = sorted(topk_ratios, reverse=True)

    prev_vecs = q_norm.clone()
    prev_C    = N

    for ratio in sorted_ratios:
        target_C = max(1, round(N * ratio))

        if target_C >= prev_C:
            centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C)
            prev_vecs = centroids
            prev_C    = centroids.shape[0]

        M_c              = fast_maxsim(centroids, doc_matrix, doc_mask)  # (K, n_docs)
        results[ratio]   = M_c.sum(0)                                    # (n_docs,)

    return results


# ==============================================================================
# METRICS
# ==============================================================================
def compute_ndcg(ranked, gt, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt):
    for r, i in enumerate(top_k):
        if i in gt: return r + 1
    return -1

def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }

def _init():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add(dst, src):
    for k in ('r1','r5','r10'): dst[k] += int(src[k])
    for k in ('n1','n5','n10'): dst[k] += float(src[k])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init()
    return store[key]

def record(key, m, domain):
    _add(_ensure(all_metrics, key), m)
    _add(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# METHOD KEYS
# ==============================================================================
ABLATION_KEYS = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]

# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

print(f"topk_ratios : {TOPK_RATIOS}  →  K = max(1, round(N_tokens * ratio))")
print("=" * 70)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # --- Load index ---
    index_path = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
    if not os.path.exists(index_path):
        print(f"  ⚠️  Not found: {index_path}"); continue
    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        fused_embeddings = saved.get('embeddings', []) if isinstance(saved, dict) else saved
        print(f"  ✅ {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ {e}"); continue

    # --- Load batch data ---
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="  Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                dfs.append(temp[[c for c in cols if c in temp.columns]])
            except Exception: pass

        batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
            columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'}
        ) if dfs else pd.DataFrame()

        df = pd.merge(batch_df_orig, batch_df_enh, on=['join_doc_name', 'layout_id'], how='left')
        df = df.sort_values(['join_doc_name', 'page_id', 'layout_id'])

        is_header = df['type'].isin(['title', 'section_header', 'header']) | \
                    df.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        df['temp_header']     = df['text'].where(is_header)
        df['current_section'] = df.groupby('join_doc_name')['temp_header'] \
                                   .ffill().fillna("General Content")

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None

        if 'image_binary' in df.columns:
            m = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[m, 'img_type'] = 'binary'
            df.loc[m, 'img_data'] = df.loc[m, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = "Section: " + df['current_section'].fillna('') + \
                           " \n Content: " + df.apply(_pick_text, axis=1)
        layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ {len(layouts_df)} layouts with image")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}"); import traceback; traceback.print_exc(); continue

    # --- Build QA pairs ---
    def calculate_iou(b1, b2):
        b1, b2 = list(b1), list(b2)
        xi, yi = max(b1[0], b2[0]), max(b1[1], b2[1])
        xa, ya = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, xa - xi) * max(0, ya - yi)
        union  = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
        return inter / union if union > 0 else 0.0

    qa_pairs   = []
    doc_lookup = {k: v for k, v in layouts_df.groupby('join_doc_name')}
    avail_docs = set(doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in avail_docs: continue
            doc_layouts = doc_lookup[target_doc].copy()
            col_page    = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            doc_layouts['safe_page'] = pd.to_numeric(
                doc_layouts[col_page], errors='coerce').fillna(-999).astype(int)
            domain = doc_data.get('domain', 'General')
            for q_item in doc_data.get('questions', []):
                gt = []
                for target in q_item.get('layout_mapping', []):
                    try:
                        tp    = int(target['page'])
                        cands = pd.concat([doc_layouts[doc_layouts['safe_page'] == tp],
                                           doc_layouts[doc_layouts['safe_page'] == tp - 1]])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], target['bbox']) > 0.5:
                                gt.append(int(idx))
                    except Exception: continue
                if gt:
                    qa_pairs.append({'question': q_item['Q'], 'gt_indices': list(set(gt)),
                                     'doc_name': target_doc, 'domain': domain})

    print(f"  ✅ {len(qa_pairs)} QA pairs")
    if not qa_pairs:
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'n_qa': 0, 'status': 'no_qa'})
        continue

    # --- Evaluate ---
    try:
        device   = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs   = len(fused_embeddings)
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []
        batch_metrics    = {k: _init() for k in ABLATION_KEYS}

        for qs in range(0, len(qa_pairs), QUERY_BATCH_SIZE):
            qe = min(qs + QUERY_BATCH_SIZE, len(qa_pairs))
            pbar = tqdm(enumerate(qa_pairs[qs:qe]), total=qe - qs,
                        desc=f"  Q[{qs}:{qe}]", leave=False)

            for qi, item in pbar:
                gq     = qs + qi
                gt     = set(item['gt_indices'])
                domain = item['domain']
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([item['question']]).to(device)
                    outputs  = model(**q_inputs)
                    if isinstance(outputs, torch.Tensor):
                        proj = outputs[0].float()
                    elif hasattr(outputs, 'last_hidden_state'):
                        proj = outputs.last_hidden_state[0].float()
                    else:
                        proj = outputs[0][0].float()
                    proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)

                # Token hợp lệ (loại special tokens)
                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()
                valid_idx = torch.where(content_mask_1d > 0)[0]
                if valid_idx.numel() == 0:
                    valid_idx = torch.where(q_inputs['attention_mask'][0] > 0)[0]

                # q_norm: (N, D) — token hợp lệ, đã L2-normalize
                q_norm = F.normalize(proj[valid_idx], dim=-1)
                N_tok  = q_norm.shape[0]

                # ------------------------------------------------------------
                # Baseline: traditional MaxSim — sum toàn bộ token
                # ------------------------------------------------------------
                M_trad      = fast_maxsim(q_norm, doc_matrix, doc_mask)  # (N, n_docs)
                trad_scores = M_trad.sum(0)
                trad_top10  = torch.topk(trad_scores, min(10, n_docs)).indices.cpu().tolist()
                m_trad      = hit_metrics(trad_top10, gt)
                _add(batch_metrics['traditional'], m_trad)
                record('traditional', m_trad, domain)

                query_row = {
                    'batch': f"{START_IDX}-{END_IDX}", 'query_id': gq,
                    'doc_name': item['doc_name'], 'domain': domain,
                    'question': item['question'],
                    'trad_r@1': m_trad['r1'],  'trad_r@5': m_trad['r5'],
                    'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
                }

                # ------------------------------------------------------------
                # Ward Hierarchical Pool — tất cả ratio trong 1 lần
                # N token → K = round(N * ratio) cụm (Ward cosine)
                # mean pool mỗi cụm → K vectors → MaxSim → sum
                # ------------------------------------------------------------
                ratio_scores = ward_pool_scores_all_ratios(
                    q_norm, doc_matrix, doc_mask, TOPK_RATIOS
                )

                for r in TOPK_RATIOS:
                    key    = f"hier_r{int(r*100)}"
                    scores = ratio_scores[r]
                    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
                    m      = hit_metrics(top10, gt)
                    _add(batch_metrics[key], m)
                    record(key, m, domain)
                    query_row.update({
                        f'{key}_r@1':     m['r1'],
                        f'{key}_r@5':     m['r5'],
                        f'{key}_r@10':    m['r10'],
                        f'{key}_ndcg@10': round(m['n10'], 4),
                    })

                batch_query_rows.append(query_row)

            gc.collect(); torch.cuda.empty_cache()
            print(f"    ✓ Q[{qs}:{qe}] done")

        all_query_results.extend(batch_query_rows)

        t   = batch_metrics['traditional']
        cnt = t['count'] or 1
        print(f"  ✅ {len(batch_query_rows)} queries — "
              f"Baseline R@10={t['r10']/cnt*100:.1f}%  nDCG@10={t['n10']/cnt:.4f}")

        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}", 'n_layouts': n_docs,
            'n_queries': len(qa_pairs),
            'trad_r10':    round(t['r10'] / cnt * 100, 2),
            'trad_ndcg10': round(t['n10'] / cnt, 4),
            'status': 'ok'
        })

    except Exception as e:
        print(f"  ❌ Eval error: {e}"); import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'status': 'error', 'error': str(e)})

    # --- Save & cleanup ---
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"),
                index=False)
        print("  ✅ Batch CSV saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask, fused_embeddings, layouts_df, qa_pairs, df
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Cleaned\n")


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING")
print("=" * 80)

try:
    if all_query_results:
        df_q = pd.DataFrame(all_query_results)
        df_q.to_csv(os.path.join(WORKING_DIR, "MASTER_queries.csv"), index=False)
        print(f"✅ {len(df_q)} queries → MASTER_queries.csv")

    # Overall table
    print(f"\n{'Method':<20} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@10':>9}")
    print("-" * 50)
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<20} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  "
              f"{m['r10']/cnt*100:6.2f}%  {m['n10']/cnt:8.4f}")

    # Per-domain breakdown
    print("\n" + "=" * 80)
    print("PER-DOMAIN — traditional vs ward per ratio")
    print("=" * 80)
    ratio_keys = [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]
    hdr = " | ".join(f"r{int(r*100):>3}" for r in TOPK_RATIOS)
    print(f"{'Domain':<22} | {'trad':>6} | {hdr}")
    print("-" * (22 + 3 + 8 + 3 + len(TOPK_RATIOS) * 8))
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        t_m = dm.get('traditional', _init()); tc = t_m['count'] or 1
        vals = []
        for key in ratio_keys:
            m_ = dm.get(key, _init()); c_ = m_['count'] or 1
            vals.append(f"{m_['n10']/c_:6.4f}")
        print(f"{domain:<22} | {t_m['n10']/tc:6.4f} | {' | '.join(vals)}")

    # Save CSVs
    summary_rows = []
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        summary_rows.append({
            'method':  key,
            'r@1':     round(m['r1']  / cnt * 100, 4),
            'r@5':     round(m['r5']  / cnt * 100, 4),
            'r@10':    round(m['r10'] / cnt * 100, 4),
            'ndcg@1':  round(m['n1']  / cnt,       6),
            'ndcg@5':  round(m['n5']  / cnt,       6),
            'ndcg@10': round(m['n10'] / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv"), index=False)

    domain_rows = []
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        row = {'domain': domain}
        for key in ABLATION_KEYS:
            m_   = dm.get(key, _init()); cnt_ = m_['count'] or 1
            row[f'{key}_ndcg10'] = round(m_['n10'] / cnt_, 6)
            row[f'{key}_r10']    = round(m_['r10'] / cnt_ * 100, 4)
        domain_rows.append(row)
    pd.DataFrame(domain_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_domain_summary.csv"), index=False)

    pd.DataFrame(all_batch_stats).to_csv(
        os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)

    print("\n✅ MASTER_ablation_summary.csv")
    print("✅ MASTER_domain_summary.csv")
    print("✅ MASTER_batch_stats.csv")

except Exception as e:
    print(f"❌ Summary error: {e}"); import traceback; traceback.print_exc()

# ==============================================================================
# VISUALIZE
# ==============================================================================
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    total_q = sum(
        row.get('n_queries', 0)
        for row in all_batch_stats if row.get('status') == 'ok'
    )

    metrics_to_plot = [
        ('n10', 'nDCG@10',   False),
        ('r10', 'Recall@10', True),
        ('n5',  'nDCG@5',    False),
        ('r1',  'Recall@1',  True),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax_idx, (mkey, title, pct_flag) in enumerate(metrics_to_plot):
        ax = axes[ax_idx]
        ax.set_title(title, fontsize=12, fontweight='normal', pad=8)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.grid(axis='x', alpha=0.15, linewidth=0.4)
        ax.spines[['top', 'right']].set_visible(False)

        # Baseline dashed line
        if total_q > 0 and 'traditional' in all_metrics:
            base_val = all_metrics['traditional'][mkey] / total_q
            if pct_flag: base_val *= 100
            ax.axhline(base_val, color='#888780', linewidth=1.8,
                       linestyle='--', label='Baseline (full ColSMoL)', zorder=2)

        # Ward pooling line
        vals = []
        for r in TOPK_RATIOS:
            key = f"hier_r{int(r*100)}"
            if key in all_metrics and total_q > 0:
                v = all_metrics[key][mkey] / total_q
                if pct_flag: v *= 100
                vals.append(v)
            else:
                vals.append(None)

        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#1DB954', linewidth=2.2,
                    marker='o', markersize=5,
                    label='Ward Hierarchical Pooling', zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda v, _: f"{v:.1f}%" if pct_flag else f"{v:.3f}")
        )
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Token Pooling with Ward Hierarchical Clustering vs Baseline\n"
        "K clusters = round(N_tokens × top-k ratio)",
        fontsize=11, y=1.02
    )
    plt.tight_layout()

    plot_path = os.path.join(WORKING_DIR, "hierarchical_pooling_ward_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot saved → {plot_path}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")

print("\n>>> BƯỚC 3 v12-HierOnly DONE")